# Car Counting System - Inference & Gradio App
## YOLO11s + ByteTrack + Line-Crossing Counting

This notebook:
1. Loads the fine-tuned model (`models/best.pt`)
2. Tests ByteTrack object tracking
3. Implements line-crossing car counting
4. Tests the counting pipeline locally
5. Provides a Gradio web interface for video upload and processing

## 1. Configuration

In [2]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

MODEL_PATH = PROJECT_ROOT / "models" / "best.pt"
VIDEO_DIR = PROJECT_ROOT / "videos"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
COUNTING_OUTPUT_DIR = OUTPUT_DIR / "counting"
TRACKING_OUTPUT_DIR = OUTPUT_DIR / "tracking"

COUNTING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRACKING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LINE_Y_POSITION = 0.5

TEST_VIDEO = list(VIDEO_DIR.glob("*.mp4"))
if TEST_VIDEO:
    TEST_VIDEO = TEST_VIDEO[0]
else:
    TEST_VIDEO = None

print(f"PROJECT_ROOT:       {PROJECT_ROOT}")
print(f"MODEL_PATH:         {MODEL_PATH}")
print(f"COUNTING_OUTPUT:    {COUNTING_OUTPUT_DIR}")
print(f"TRACKING_OUTPUT:    {TRACKING_OUTPUT_DIR}")
print(f"TEST_VIDEO:         {TEST_VIDEO}")
print(f"LINE_Y_POSITION:    {LINE_Y_POSITION} (fraction of frame height)")

PROJECT_ROOT:       d:\university\term8\car-counting-system
MODEL_PATH:         d:\university\term8\car-counting-system\models\best.pt
COUNTING_OUTPUT:    d:\university\term8\car-counting-system\outputs\counting
TRACKING_OUTPUT:    d:\university\term8\car-counting-system\outputs\tracking
TEST_VIDEO:         d:\university\term8\car-counting-system\videos\Vehicle Dataset Sample 2.mp4
LINE_Y_POSITION:    0.5 (fraction of frame height)


## 2. Import Libraries & CUDA Check

In [3]:
import torch
import cv2
import numpy as np
import tempfile
import shutil
import time
from pathlib import Path
from datetime import datetime
from ultralytics import YOLO

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU")

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print(f"DEVICE: {DEVICE}")

try:
    import gradio as gr
    print("Gradio installed.")
except ImportError:
    print("WARNING: Gradio not installed. Run: pip install gradio")
    gr = None

CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
DEVICE: 0


d:\university\term8\car-counting-system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Gradio installed.


## 3. Load Trained Model

In [4]:
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Model not found at {MODEL_PATH}.\n"
        "Run car_training.ipynb first to train and save the model."
    )

model = YOLO(str(MODEL_PATH))
print(f"Model loaded from: {MODEL_PATH}")
print(f"Model names: {model.names}")

CAR_KEYWORDS = ["sedan", "suv", "pickup"]
car_class_ids = [
    i for i, name in model.names.items()
    if any(keyword in name.lower() for keyword in CAR_KEYWORDS)
]

if not car_class_ids:
    raise ValueError(
        f"No car classes found in model names: {model.names}.\n"
        f"Expected keywords: {CAR_KEYWORDS}"
    )

car_class_names = {i: model.names[i] for i in car_class_ids}
print(f"Car class IDs for counting: {car_class_ids}")
print(f"Car class names: {car_class_names}")

Model loaded from: d:\university\term8\car-counting-system\models\best.pt
Model names: {0: 'Bus', 1: 'Motorcycle', 2: 'Pickup', 3: 'SUV', 4: 'Sedan', 5: 'Truck', 6: 'Van'}
Car class IDs for counting: [2, 3, 4]
Car class names: {2: 'Pickup', 3: 'SUV', 4: 'Sedan'}


## 4. Test ByteTrack Tracking

In [5]:
TRACKING_TEST_VIDEO = None
for ext in ["*.mp4", "*.avi", "*.mov"]:
    candidates = list(VIDEO_DIR.glob(ext))
    if candidates:
        TRACKING_TEST_VIDEO = candidates[0]
        break

if TRACKING_TEST_VIDEO and TRACKING_TEST_VIDEO.exists():
    print(f"Testing ByteTrack on: {TRACKING_TEST_VIDEO.name}")
    tracking_output = TRACKING_OUTPUT_DIR / f"tracking_test_{TRACKING_TEST_VIDEO.stem}.mp4"

    cap = cv2.VideoCapture(str(TRACKING_TEST_VIDEO))
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"  Resolution: {width}x{height}, FPS: {fps:.1f}, Frames: {total_frames}")

    fourcc = cv2.VideoWriter_fourcc(*"avc1")
    out = cv2.VideoWriter(str(tracking_output), fourcc, fps, (width, height))

    frame_count = 0
    max_test_frames = 300

    while cap.isOpened() and frame_count < max_test_frames:
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1
        results = model.track(
            frame, persist=True, tracker="bytetrack.yaml",
            device=DEVICE, classes=car_class_ids, verbose=False
        )

        annotated = results[0].plot()
        out.write(annotated)

        if frame_count % 30 == 0:
            print(f"  Processed {frame_count}/{min(total_frames, max_test_frames)} frames")

    cap.release()
    out.release()
    print(f"Tracking test output saved to: {tracking_output}")
else:
    print("No test video found. Skipping tracking test.")

Testing ByteTrack on: Vehicle Dataset Sample 2.mp4
  Resolution: 1280x720, FPS: 30.0, Frames: 305
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   -------------- ------------------------- 0.5/1.5 MB 1.5 MB/s eta 0:00:01
   ---------------------------- ----------- 1.0/1.5 MB 1.7 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 1.8 MB/s  0:00:00

requirements: AutoUpdate success  5.6s
WARNING requirements: Restart runtime or rerun command for updates to take effect

  Processed 30/300 frames
  Processed 60/300 frames
  Processed 90/300 frames
  Processed 120/300 frames
  Processed 150/300 frames
  Processed 180/300 frames
  Processed 210/300 frames
  Processed 240/300 frames
  Processed 270/300 frames
  Processed 300/3

## 5. Counting Logic - Line-Crossing Detection

How it works:
1. A horizontal virtual line is drawn at `LINE_Y_POSITION * frame_height`
2. For each tracked car, the center point is calculated from the bounding box
3. When a car's center crosses from above to below the line, it is counted
4. Each tracking ID is counted only once (stored in `counted_ids`)

Only one direction is counted (top to bottom) for simplicity.

In [6]:
def count_cars(video_path, model, car_ids, line_y_pos=0.5, device="cpu", output_dir=None):
    """
    Process a video: detect, track, and count cars crossing a virtual line.

    Args:
        video_path: Path to input video file.
        model: Loaded YOLO model.
        car_ids: List of class IDs considered as cars.
        line_y_pos: Fraction of frame height for the counting line (0-1).
        device: Device for inference ('cpu', 0, etc.).
        output_dir: Directory to save the output video.

    Returns:
        Tuple of (output_video_path, total_car_count).
    """
    video_path = Path(video_path)
    if not video_path.exists():
        raise FileNotFoundError(f"Video not found: {video_path}")

    if output_dir is None:
        output_dir = COUNTING_OUTPUT_DIR
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_name = f"car_counting_{video_path.stem}_{timestamp}.mp4"
    output_path = output_dir / output_name

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        fps = 30.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    LINE_Y = int(height * line_y_pos)

    fourcc = cv2.VideoWriter_fourcc(*"avc1")
    out = cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))

    prev_centers = {}
    counted_ids = set()
    total_count = 0
    frame_idx = 0

    print(f"Processing: {video_path.name}")
    print(f"  Resolution: {width}x{height}, FPS: {fps:.1f}, Frames: {total_frames}")
    print(f"  Counting line at Y={LINE_Y} ({line_y_pos*100:.0f}% of height)")
    print(f"  Car class IDs: {car_ids}")

    start_time = time.time()

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_idx += 1

        results = model.track(
            frame, persist=True, tracker="bytetrack.yaml",
            device=device, classes=car_ids, verbose=False
        )

        annotated = results[0].plot()

        boxes = results[0].boxes
        if boxes is not None and boxes.id is not None:
            track_ids = boxes.id.int().cpu().tolist()
            xyxy = boxes.xyxy.cpu().tolist()

            for track_id, (x1, y1, x2, y2) in zip(track_ids, xyxy):
                cx = int((x1 + x2) / 2)
                cy = int((y1 + y2) / 2)

                cv2.circle(annotated, (cx, cy), 4, (0, 255, 255), -1)
                cv2.putText(annotated, f"ID:{track_id}", (cx + 6, cy - 6),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)

                prev_cy = prev_centers.get(track_id, cy)

                if prev_cy < LINE_Y and cy >= LINE_Y:
                    if track_id not in counted_ids:
                        total_count += 1
                        counted_ids.add(track_id)

                prev_centers[track_id] = cy

        cv2.line(annotated, (0, LINE_Y), (width, LINE_Y), (0, 0, 255), 2)
        cv2.putText(annotated, f"CARS COUNTED: {total_count}", (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)

        out.write(annotated)

        if frame_idx % 100 == 0:
            elapsed = time.time() - start_time
            progress = frame_idx / total_frames * 100 if total_frames > 0 else 0
            print(f"  Frame {frame_idx}/{total_frames} ({progress:.0f}%), "
                  f"Count: {total_count}, Elapsed: {elapsed:.1f}s")

    cap.release()
    out.release()

    elapsed = time.time() - start_time
    print(f"\nProcessing complete!")
    print(f"  Total frames processed: {frame_idx}")
    print(f"  Total cars counted:     {total_count}")
    print(f"  Time elapsed:           {elapsed:.1f}s")
    print(f"  Output:                 {output_path}")

    return str(output_path), total_count

## 6. Test Counting Locally

In [7]:
if TEST_VIDEO and TEST_VIDEO.exists():
    output_video_path, car_count = count_cars(
        video_path=TEST_VIDEO,
        model=model,
        car_ids=car_class_ids,
        line_y_pos=LINE_Y_POSITION,
        device=DEVICE,
        output_dir=COUNTING_OUTPUT_DIR,
    )
    print(f"\nResult: {car_count} cars counted")
else:
    print("No test video available. Counting test skipped.")
    print("Upload a video using the Gradio interface below, or place a .mp4 file in the videos/ directory.")

Processing: Vehicle Dataset Sample 2.mp4
  Resolution: 1280x720, FPS: 30.0, Frames: 305
  Counting line at Y=360 (50% of height)
  Car class IDs: [2, 3, 4]
  Frame 100/305 (33%), Count: 7, Elapsed: 2.1s
  Frame 200/305 (66%), Count: 17, Elapsed: 4.0s
  Frame 300/305 (98%), Count: 27, Elapsed: 5.9s

Processing complete!
  Total frames processed: 302
  Total cars counted:     27
  Time elapsed:           6.0s
  Output:                 d:\university\term8\car-counting-system\outputs\counting\car_counting_Vehicle Dataset Sample 2_20260728_070808.mp4

Result: 27 cars counted


## 7. Gradio Interface

Upload a traffic video to detect, track, and count cars using YOLO11s and ByteTrack.

In [8]:
if gr is None:
    raise ImportError("Gradio is not installed. Run: pip install gradio")

def process_video(video_file):
    """Gradio handler: takes uploaded video, returns processed video and count."""
    if video_file is None:
        return None, "No video uploaded."

    try:
        output_path, car_count = count_cars(
            video_path=video_file,
            model=model,
            car_ids=car_class_ids,
            line_y_pos=LINE_Y_POSITION,
            device=DEVICE,
            output_dir=COUNTING_OUTPUT_DIR,
        )
        count_text = f"Total cars counted: {car_count}"
        return output_path, count_text
    except Exception as e:
        return None, f"Error processing video: {str(e)}"

with gr.Blocks(title="Car Counting System") as demo:
    gr.Markdown("# \U0001f697 Car Counting System")
    gr.Markdown(
        "Upload a traffic video to detect, track, and count cars "
        "using **YOLO11s** and **ByteTrack**."
    )

    with gr.Row():
        with gr.Column():
            input_video = gr.Video(label="Upload Traffic Video")
            process_btn = gr.Button("Process Video", variant="primary")

        with gr.Column():
            output_video = gr.Video(label="Processed Video", autoplay=False)
            count_display = gr.Textbox(label="Car Count", value="Waiting for video...")

    process_btn.click(
        fn=process_video,
        inputs=[input_video],
        outputs=[output_video, count_display],
    )

    gr.Markdown(
        "### How it works\n"
        "1. Upload a traffic video\n"
        "2. Click Process Video\n"
        "3. YOLO11s detects cars\n"
        "4. ByteTrack assigns tracking IDs\n"
        "5. A virtual line detects crossings\n"
        "6. Each car is counted once\n"
        "7. Processed video and count are displayed"
    )

print("Gradio interface defined. Launch with the next cell.")

Gradio interface defined. Launch with the next cell.


## 8. Launch Gradio (Local)

In [9]:
if gr is not None:
    demo.launch()
else:
    print("Gradio not available.")

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Processing: Vehicle Dataset Sample 2.mp4
  Resolution: 1280x720, FPS: 30.0, Frames: 305
  Counting line at Y=360 (50% of height)
  Car class IDs: [2, 3, 4]
  Frame 100/305 (33%), Count: 7, Elapsed: 2.3s
  Frame 200/305 (66%), Count: 17, Elapsed: 4.2s
  Frame 300/305 (98%), Count: 27, Elapsed: 6.1s

Processing complete!
  Total frames processed: 302
  Total cars counted:     27
  Time elapsed:           6.2s
  Output:                 d:\university\term8\car-counting-system\outputs\counting\car_counting_Vehicle Dataset Sample 2_20260728_070935.mp4


Traceback (most recent call last):
  File "d:\university\term8\car-counting-system\.venv\Lib\site-packages\gradio\queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "d:\university\term8\car-counting-system\.venv\Lib\site-packages\gradio\route_utils.py", line 393, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "d:\university\term8\car-counting-system\.venv\Lib\site-packages\gradio\blocks.py", line 2293, in process_api
    data = await self.postprocess_data(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        block_fn, result["prediction"], state
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "d:\university\term8\car-counting-system\.venv\Lib\site-packages\gradio\blocks.py", line 2065, in postprocess_data
    await processing_utils.asy

Processing: Vehicle Dataset Sample 2.mp4
  Resolution: 1280x720, FPS: 30.0, Frames: 305
  Counting line at Y=360 (50% of height)
  Car class IDs: [2, 3, 4]
  Frame 100/305 (33%), Count: 7, Elapsed: 2.3s
  Frame 200/305 (66%), Count: 17, Elapsed: 4.2s
  Frame 300/305 (98%), Count: 27, Elapsed: 6.1s

Processing complete!
  Total frames processed: 302
  Total cars counted:     27
  Time elapsed:           6.2s
  Output:                 d:\university\term8\car-counting-system\outputs\counting\car_counting_Vehicle Dataset Sample 2_20260728_070953.mp4


Traceback (most recent call last):
  File "d:\university\term8\car-counting-system\.venv\Lib\site-packages\gradio\queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "d:\university\term8\car-counting-system\.venv\Lib\site-packages\gradio\route_utils.py", line 393, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "d:\university\term8\car-counting-system\.venv\Lib\site-packages\gradio\blocks.py", line 2293, in process_api
    data = await self.postprocess_data(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        block_fn, result["prediction"], state
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "d:\university\term8\car-counting-system\.venv\Lib\site-packages\gradio\blocks.py", line 2065, in postprocess_data
    await processing_utils.asy

Processing: Vehicle Dataset Sample 2.mp4
  Resolution: 1280x720, FPS: 30.0, Frames: 305
  Counting line at Y=360 (50% of height)
  Car class IDs: [2, 3, 4]
  Frame 100/305 (33%), Count: 7, Elapsed: 2.2s
  Frame 200/305 (66%), Count: 17, Elapsed: 4.1s
  Frame 300/305 (98%), Count: 27, Elapsed: 6.0s

Processing complete!
  Total frames processed: 302
  Total cars counted:     27
  Time elapsed:           6.0s
  Output:                 d:\university\term8\car-counting-system\outputs\counting\car_counting_Vehicle Dataset Sample 2_20260728_071011.mp4


Traceback (most recent call last):
  File "d:\university\term8\car-counting-system\.venv\Lib\site-packages\gradio\queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "d:\university\term8\car-counting-system\.venv\Lib\site-packages\gradio\route_utils.py", line 393, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "d:\university\term8\car-counting-system\.venv\Lib\site-packages\gradio\blocks.py", line 2293, in process_api
    data = await self.postprocess_data(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        block_fn, result["prediction"], state
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "d:\university\term8\car-counting-system\.venv\Lib\site-packages\gradio\blocks.py", line 2065, in postprocess_data
    await processing_utils.asy

## Academic Evaluation: Counting Accuracy

To evaluate counting accuracy, compare the system's count against
a manually counted ground truth:

```text
Counting Accuracy = (Correctly Counted / Actual Vehicles) x 100
```

Run the cell below after manually counting vehicles in a test video.

In [ ]:
ACTUAL_VEHICLES = None
SYSTEM_COUNT = None

if ACTUAL_VEHICLES is not None and SYSTEM_COUNT is not None:
    accuracy = (SYSTEM_COUNT / ACTUAL_VEHICLES) * 100 if ACTUAL_VEHICLES > 0 else 0
    print("=" * 40)
    print("COUNTING ACCURACY EVALUATION")
    print("=" * 40)
    print(f"Actual vehicles (manual):   {ACTUAL_VEHICLES}")
    print(f"System counted:              {SYSTEM_COUNT}")
    print(f"Counting accuracy:           {accuracy:.1f}%")
    print("=" * 40)
else:
    print("Set ACTUAL_VEHICLES to the manually counted number")
    print("and SYSTEM_COUNT to the result from test counting above.")

Exception in callback _ProactorBasePipeTransport._call_connection_lost()
handle: <Handle _ProactorBasePipeTransport._call_connection_lost()>
Traceback (most recent call last):
  File "C:\Users\M\AppData\Local\Python\pythoncore-3.14-64\Lib\asyncio\events.py", line 94, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\M\AppData\Local\Python\pythoncore-3.14-64\Lib\asyncio\proactor_events.py", line 165, in _call_connection_lost
    self._sock.shutdown(socket.SHUT_RDWR)
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host


Processing: Vehicle Dataset Sample 2.mp4
  Resolution: 1280x720, FPS: 30.0, Frames: 305
  Counting line at Y=360 (50% of height)
  Car class IDs: [2, 3, 4]
  Frame 100/305 (33%), Count: 7, Elapsed: 3.2s
  Frame 200/305 (66%), Count: 17, Elapsed: 6.5s
  Frame 300/305 (98%), Count: 27, Elapsed: 9.7s

Processing complete!
  Total frames processed: 302
  Total cars counted:     27
  Time elapsed:           9.7s
  Output:                 d:\university\term8\car-counting-system\outputs\counting\car_counting_Vehicle Dataset Sample 2_20260728_080622.mp4


Traceback (most recent call last):
  File "d:\university\term8\car-counting-system\.venv\Lib\site-packages\gradio\queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "d:\university\term8\car-counting-system\.venv\Lib\site-packages\gradio\route_utils.py", line 393, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "d:\university\term8\car-counting-system\.venv\Lib\site-packages\gradio\blocks.py", line 2293, in process_api
    data = await self.postprocess_data(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        block_fn, result["prediction"], state
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "d:\university\term8\car-counting-system\.venv\Lib\site-packages\gradio\blocks.py", line 2065, in postprocess_data
    await processing_utils.asy